# Poisson Regression: Predicting Goals in Matches

**Chapter 6: Regression Techniques for Soccer Analytics**

## Learning Objectives

- Understand why Poisson regression is ideal for count data
- Learn the difference between linear and Poisson regression
- Build a Poisson model to predict goals scored
- Use statsmodels for GLM (Generalized Linear Models)
- Interpret coefficients on a logarithmic scale
- Visualize non-linear relationships


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## Why Poisson Regression for Goals?

Linear regression is great for continuous values, but **goals are counts**. Here's why Poisson is better:

| Issue | Linear Regression | Poisson Regression |
|-------|-------------------|-------------------|
| **Fractional values** | Can predict 2.7 goals | Only predicts whole numbers |
| **Negative values** | Can predict -1 goals | Only predicts non-negative |
| **Relationship** | Assumes linear | Handles non-linear patterns |

**Poisson regression** is a type of **Generalized Linear Model (GLM)** specifically designed for count data.


## The Poisson Distribution

**Key assumptions:**
1. Events (goals) occur at a constant average rate
2. Events are independent
3. Two events cannot occur at exactly the same instant

This fits soccer matches well! Goals are discrete events that occur independently throughout a match.


## Load Data: Goals Scored

We'll predict the number of goals a team scores based on shots on target.


In [ ]:
# Simulated data for goals scored
data = {
    'Team': [f'Team {i}' for i in range(1, 21)],
    'ShotsOnTarget': [5, 8, 12, 4, 6, 9, 10, 3, 7, 11, 5, 8, 12, 4, 6, 9, 10, 3, 7, 11],
    'Goals': [1, 2, 3, 0, 1, 2, 2, 0, 1, 3, 1, 2, 4, 1, 1, 2, 3, 0, 2, 3]
}
goals_df = pd.DataFrame(data)

print(\"Goals Dataset:\")
print(goals_df.head(10))
print(f\"\
Dataset shape: {goals_df.shape}\")
print(f\"\
Goals distribution:\")
print(goals_df['Goals'].value_counts().sort_index())


## Visualize the Data


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, s=100)
plt.title('Goals vs. Shots on Target', fontsize=14)
plt.xlabel('Shots on Target')
plt.ylabel('Number of Goals')
plt.tight_layout()
plt.show()

print(\"Notice: Goals are discrete (0, 1, 2, 3...), not continuous.\")
print(\"This is why we need Poisson regression!\")


## Build the Poisson Regression Model

We'll use **statsmodels** which provides excellent support for GLMs and detailed statistical summaries.


In [ ]:
# Build the Poisson model using formula notation
formula = \"Goals ~ ShotsOnTarget\"
model = smf.poisson(formula, data=goals_df).fit()

# Print model summary
print(model.summary())


## Interpret the Coefficients

**Important:** Poisson regression uses a **log link function**, so coefficients are on a logarithmic scale.

To interpret:
- **Exponentiate the coefficient**: `exp(coef)` gives the multiplicative effect
- If `coef = 0.15`, then `exp(0.15) ≈ 1.16`
- **Interpretation:** Each additional shot on target multiplies expected goals by 1.16 (a 16% increase)


In [ ]:
# Extract and interpret coefficients
coef_shots = model.params['ShotsOnTarget']
multiplicative_effect = np.exp(coef_shots)

print(f\"Coefficient for ShotsOnTarget: {coef_shots:.4f}\")
print(f\"Multiplicative effect: {multiplicative_effect:.4f}\")
print(f\"\
Interpretation:\")
print(f\"Each additional shot on target increases expected goals by {(multiplicative_effect-1)*100:.1f}%\")


## Visualize the Poisson Model

Unlike linear regression, Poisson creates a **curved prediction line**.


In [ ]:
# Get model predictions
goals_df['predicted_goals'] = model.predict(goals_df)

plt.figure(figsize=(10, 6))

# Plot actual goals
sns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, label='Actual Goals', s=100)

# Plot predicted goals (curved line)
sorted_df = goals_df.sort_values('ShotsOnTarget')
plt.plot(sorted_df['ShotsOnTarget'], sorted_df['predicted_goals'], 
         color='red', linewidth=3, linestyle='--', label='Predicted Goals (Poisson)')

plt.title('Poisson Regression: Goals vs. Shots on Target', fontsize=14)
plt.xlabel('Shots on Target')
plt.ylabel('Number of Goals')
plt.legend()
plt.tight_layout()
plt.show()

print(\"Notice: The prediction line curves upward!\")
print(\"This captures the non-linear relationship better than a straight line.\")


## Make Predictions

Predict goals for teams with different shot counts.


In [ ]:
# Create new data for prediction
new_teams = pd.DataFrame({
    'ShotsOnTarget': [3, 6, 9, 12, 15]
})

# Predict expected goals
predictions = model.predict(new_teams)

print(\"Expected Goals Predictions:\")
for shots, goals in zip(new_teams['ShotsOnTarget'], predictions):
    print(f\"  {shots} shots on target → {goals:.2f} expected goals\")


## Compare: Linear vs. Poisson

Let's see the difference between linear and Poisson regression.


In [ ]:
from sklearn.linear_model import LinearRegression

# Build linear model
X = goals_df[['ShotsOnTarget']]
y = goals_df['Goals']
linear_model = LinearRegression()
linear_model.fit(X, y)
goals_df['linear_pred'] = linear_model.predict(X)

# Plot comparison
plt.figure(figsize=(12, 6))
sns.scatterplot(x='ShotsOnTarget', y='Goals', data=goals_df, s=100, label='Actual')

sorted_df = goals_df.sort_values('ShotsOnTarget')
plt.plot(sorted_df['ShotsOnTarget'], sorted_df['predicted_goals'], 
         color='red', linewidth=3, label='Poisson', linestyle='--')
plt.plot(sorted_df['ShotsOnTarget'], sorted_df['linear_pred'], 
         color='blue', linewidth=3, label='Linear', linestyle='-.')

plt.title('Poisson vs. Linear Regression', fontsize=14)
plt.xlabel('Shots on Target')
plt.ylabel('Goals')
plt.legend()
plt.tight_layout()
plt.show()

print(\"Key Difference:\")
print(\"- Linear: Straight line (can predict negative/fractional goals)\")
print(\"- Poisson: Curved line (always non-negative, better for counts)\")


## Model Evaluation

For count data, we can use metrics like **Mean Absolute Error (MAE)**.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate errors
mae_poisson = mean_absolute_error(goals_df['Goals'], goals_df['predicted_goals'])
mae_linear = mean_absolute_error(goals_df['Goals'], goals_df['linear_pred'])

print(\"Model Performance Comparison:\")
print(f\"  Poisson MAE: {mae_poisson:.3f}\")
print(f\"  Linear MAE:  {mae_linear:.3f}\")

if mae_poisson < mae_linear:
    print(\"\
Poisson regression performs better for this count data!\")
else:
    print(\"\
Linear regression performs similarly (but Poisson is still more appropriate for counts)\")


## Summary

In this notebook, we:

1. ✅ Learned why Poisson regression is ideal for count data
2. ✅ Built a Poisson model to predict goals from shots on target
3. ✅ Used statsmodels for GLM modeling
4. ✅ Interpreted coefficients on a logarithmic scale
5. ✅ Visualized the non-linear relationship
6. ✅ Compared Poisson vs. linear regression

## Key Takeaways

- **Poisson regression** is designed for count data (goals, cards, corners)
- Uses a **log link function** → coefficients need exponentiation
- Produces **curved predictions** that respect count data properties
- Always **non-negative** predictions
- Better than linear regression for discrete outcomes

## Next Steps

In the next notebook, we'll explore **K-Nearest Neighbors (KNN) Regression** for finding similar players!


## Practice Exercises

1. **Multiple Predictors**: Add features like possession, xG, or opponent strength
2. **Match Outcome Prediction**: Predict goals for both teams and determine winner
3. **Poisson Distribution**: Plot the Poisson distribution for different lambda values
4. **Real Match Data**: Apply to actual match data from StatsBomb or similar
5. **Overdispersion Check**: Research and test if the data shows overdispersion
